In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Tuning Recall and Latency with Target Recall in Agent Retrieval

This notebook demonstrates how the **`target_recall`** search parameter in
**[Agent Retrieval](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)**
(formerly [Vector Search 2.0](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)) on [Gemini Enterprise Agent Platform](https://docs.cloud.google.com/gemini-enterprise-agent-platform) affects result quality and query latency on a 3-million-item index.

You do not need a Google Cloud project to run this notebook. Every query is sent to the public
[Agent Retrieval interactive demo](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/try-it),
so there is no infrastructure to create, wait for, or clean up.

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fembeddings%2Fagent-retrieval-target-recall.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/embeddings/agent-retrieval-target-recall.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>


| Author(s) |
| --- |
| [Kaz Sato](https://github.com/kazunori279) |

## Overview

### Default behavior is usually recommended

Agent Retrieval automatically balances recall against latency for every [Approximate Nearest Neighbor (ANN)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) search. If you do not specify `target_recall`, the service selects search settings optimized for high quality and speed based on your index structure. For most applications, this default is the right choice, and it continues to improve automatically as the underlying engine evolves.

Use `target_recall` when you need direct control over this trade-off.

### How target_recall works

Specify your desired recall target as a float between `0.0` and `1.0`:

- **`0.95`** - Retrieve approximately 95% of the nearest neighbors that an exhaustive (brute-force) search would return.

Previously, tuning this trade-off required adjusting low-level indexing knobs like [`search_leaves_pct` and `initial_candidate_count`](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/configuring-indexes), which required intimate knowledge of internal index structures. `target_recall` allows you to express your intent directly in terms of target recall, and Agent Retrieval translates it to index traversal parameters using calibration data collected during index construction.

Achieved recall is best-effort. The actual recall achieved depends on your dataset and individual query distribution, which is why Part 3 of this notebook benchmarks empirical recall and latency directly.

### When to tune it

| Scenario | Recommended action |
|---|---|
| You require higher result quality than default settings provide | Increase `target_recall` toward `1.0` |
| You prioritize lower latency or higher throughput (QPS) over marginal recall gains | Lower `target_recall` to reduce search traversal effort |
| You are satisfied with default recall and latency | Leave `target_recall` unset |

### Key requirement: Clustered ANN index

`target_recall` applies to clustered [ANN indexes](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/indexes/indexes), which Agent Retrieval builds when a [Collection](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/collections/collections) exceeds approximately **100,000 embeddings**. For smaller collections, the service uses brute-force exact search, where recall is always 100% and search effort cannot be tuned. On small collections, setting `target_recall` is accepted but has no effect.

Both **performance-optimized** and **storage-optimized** index tiers support `target_recall`. Achieved recall is always best-effort, not a hard guarantee.

This notebook uses a 3-million-item multimodal index to illustrate these dynamics clearly.

### What you will do

1. Connect to the public demo endpoint (no GCP project or credentials required).
2. Execute a single query across multiple `target_recall` values to observe result differences.
3. Benchmark recall and latency across multiple queries and visualize the Pareto frontier.
4. Review the REST API request syntax for integration into your own projects.


---

# Part 1: Connect to the demo endpoint

The only required library is `requests` (along with `pandas` and `matplotlib` for analysis and charting). No authentication or API keys are required.


In [ ]:
%pip install --upgrade --quiet requests pandas matplotlib tqdm

print("Ready.")


The endpoint powers the
[Agent Retrieval interactive demo](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/try-it)
and serves several Mercari product datasets. We use the **multimodal** dataset:

- **3 million items:** Well above the threshold required for clustered ANN indexing.
- **Separation of query embedding and search:** The endpoint generates query embeddings and performs vector search in separate steps, reporting both timings independently. This distinction is critical: generating a query embedding typically takes hundreds of milliseconds, whereas the vector search itself takes tens of milliseconds. Measuring end-to-end latency would completely obscure the vector search performance changes.

All benchmark results below report **server-side vector search time only**, excluding query embedding generation and network transit time.


In [ ]:
from typing import Any

import requests

DEMO_ENDPOINT = "https://ac-web2-nhhfh7g7iq-uc.a.run.app/api/query"
DATASET_ID = "mercari3m_multimodal"  # @param {type:"string"}
TOP_K = 10


def search(
    query: str,
    target_recall: float | None = None,
    top_k: int = TOP_K,
    dataset_id: str = DATASET_ID,
) -> dict[str, Any]:
    """Run one search against the demo endpoint.

    Args:
      query: Natural language query.
      target_recall: Desired recall in [0, 1], or None to let the service choose.
      top_k: Number of results to return.
      dataset_id: Which demo dataset to search.

    Returns:
      A dict with "ids", "names", "search_ms" (server-side search time, excluding
      query embedding and network) and "embed_ms" (the query embedding time that
      search_ms deliberately leaves out).
    """
    payload: dict[str, Any] = {
        "query": query,
        "dataset_id": dataset_id,
        "use_semantic_search": True,
        "use_text_search": False,
        "rows": top_k,
    }
    if target_recall is not None:
        if not 0.0 <= target_recall <= 1.0:
            raise ValueError("target_recall must be between 0.0 and 1.0")
        payload["target_recall"] = target_recall

    response = requests.post(DEMO_ENDPOINT, json=payload, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(
            f"Search failed ({response.status_code}): {response.text[:300]}"
        )
    body = response.json()

    return {
        "ids": [item["id"] for item in body["items"]],
        "names": [item.get("name", "") for item in body["items"]],
        # Server-reported timings, so network latency is excluded too. The two are
        # kept apart because embedding dwarfs search and would otherwise hide it.
        "search_ms": body["latencies"]["query"] * 1000,
        "embed_ms": body["latencies"]["gen_query_emb"] * 1000,
        "applied_target_recall": body.get("applied_target_recall"),
    }


# Confirm the endpoint is reachable and understands target_recall.
probe = search("camera", target_recall=0.9, top_k=3)
if probe["applied_target_recall"] != 0.9:
    raise RuntimeError(
        "The demo endpoint ignored target_recall. It is probably running an older "
        "build that predates the parameter."
    )
print(f"Connected. Example result: {probe['names'][0]!r}")
print(
    f"  query embedding: {probe['embed_ms']:6.1f} ms  (excluded from the measurements below)"
)
print(f"  search:          {probe['search_ms']:6.1f} ms  (what we measure)")

---

# Part 2: Inspecting query results across target recall levels

A low `target_recall` setting directs the index to examine minimal candidates, whereas a high setting forces a thorough search. Both evaluate the same query against the same index.

Individual query latencies are omitted here due to single-request network and cache variances. Part 3 conducts structured, repeatable latency benchmarks.


In [ ]:
EXAMPLE_QUERY = "vintage leather camera bag"  # @param {type:"string"}

reference = search(EXAMPLE_QUERY, target_recall=1.0, top_k=5)
print(f"Query: {EXAMPLE_QUERY!r}\n")
print("target_recall=1.0 (most thorough):")
for rank, name in enumerate(reference["names"], 1):
    print(f"  {rank}. {name[:70]}")

for recall_target in [0.1, 0.5, 0.9]:
    result = search(EXAMPLE_QUERY, target_recall=recall_target, top_k=5)
    overlap = len(set(result["ids"]) & set(reference["ids"]))
    print(
        f"\ntarget_recall={recall_target} ({overlap}/5 of the thorough results kept):"
    )
    for rank, name in enumerate(result["names"], 1):
        marker = " " if result["ids"][rank - 1] in reference["ids"] else "*"
        print(f"  {rank}.{marker}{name[:70]}")

print("\n* = not in the target_recall=1.0 result")

---

# Part 3: Benchmark recall and latency trade-offs

### Measuring recall

Recall evaluates search results against true nearest neighbors. For a 3-million-item index, exhaustive brute-force search is not directly available via the API (which restricts exact kNN to under 100,000 items). Therefore, we use **`target_recall = 1.0` as the baseline reference**—representing the most exhaustive search the index can perform—and evaluate how many of those top results are retained at lower recall targets:

$$\text{Recall@K} \approx \frac{|\text{result at } t \;\cap\; \text{result at } 1.0|}{K}$$

These numbers represent relative recall against the highest-effort ANN search. On your private datasets where you have direct access to raw vectors, you can compute exact ground truth offline.

### Benchmarking methodology

- **Fine-grained sampling in the non-linear region:** Because search latency and recall scale non-linearly—rising sharply between `0.9` and `1.0`—we sample this critical range in fine-grained increments (`0.90`, `0.92`, `0.94`, `0.96`, `0.98`, `0.99`, `1.0`) while taking broader samples across lower settings (`0.1`, `0.5`, `0.8`).
- **Randomized query order:** Executing targets strictly from lowest to highest would measure higher targets against a warmed server cache, creating an artificial latency trend. Shuffling the execution plan prevents cache warm-up bias.
- **Warm-up query and median aggregation:** An initial untimed request primes the connection, and median values are reported across repeated runs to minimize outlier variance.


In [ ]:
import random
import statistics

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

QUERIES = [
    "vintage leather camera bag",
    "running shoes",
    "coffee mug",
    "wool sweater",
    "gold necklace",
    "gaming keyboard",
]
# Fine-grained targets from 0.9 to 1.0 to capture the non-linear curve
RECALL_TARGETS = [0.1, 0.5, 0.8, 0.9, 0.92, 0.94, 0.96, 0.98, 0.99, 1.0]
REPEATS = 3

# 1. Reference result per query: the most thorough search the index will do.
print(f"Building reference results for {len(QUERIES)} queries...")
reference_ids = {
    query: set(search(query, target_recall=1.0)["ids"])
    for query in tqdm(QUERIES, desc="Reference queries")
}

# 2. Warm up so the first timed call is not penalized.
search(QUERIES[0], target_recall=0.9)

# 3. Shuffle every (query, target) pair so warm-up cannot look like a trend.
plan = [(q, t) for q in QUERIES for t in RECALL_TARGETS for _ in range(REPEATS)]
random.seed(42)
random.shuffle(plan)

latencies: dict[float, list[float]] = {t: [] for t in RECALL_TARGETS}
recalls: dict[float, list[float]] = {t: [] for t in RECALL_TARGETS}
embed_times: list[float] = []
scored: set[tuple[str, float]] = set()

print(f"Running {len(plan)} timed queries...")
for query, recall_target in tqdm(plan, desc="Timed queries"):
    result = search(query, target_recall=recall_target)
    latencies[recall_target].append(result["search_ms"])
    embed_times.append(result["embed_ms"])

    # Score recall once per (query, target); the repeats only contribute timings.
    if (query, recall_target) in scored:
        continue
    scored.add((query, recall_target))
    kept = set(result["ids"]) & reference_ids[query]
    recalls[recall_target].append(len(kept) / TOP_K)

df_metrics = pd.DataFrame(
    [
        {
            "target_recall": t,
            "achieved_recall": statistics.mean(recalls[t]),
            "median_search_ms": statistics.median(latencies[t]),
        }
        for t in RECALL_TARGETS
    ]
)

median_embed_ms = statistics.median(embed_times)
search_lo = df_metrics["median_search_ms"].min()
search_hi = df_metrics["median_search_ms"].max()
print()
print(
    f"Query embedding: {median_embed_ms:.0f} ms per call, excluded from the table below."
)
print(f"Search itself:   {search_lo:.0f}-{search_hi:.0f} ms across the targets.")
print("Measured end to end, the embedding call would have hidden the effect entirely.")
df_metrics


The plots below compare search cost (median search time in ms) against retrieved result quality (empirical recall@K).


In [ ]:
fig, (ax_latency, ax_recall) = plt.subplots(1, 2, figsize=(12, 4))

ax_recall.plot(
    df_metrics["target_recall"],
    df_metrics["achieved_recall"],
    marker="s",
    color="#34a853",
    label="Achieved",
)
ax_recall.plot([0, 1], [0, 1], linestyle="--", color="#9aa0a6", label="Requested")
ax_recall.set_xlabel("Requested target_recall")
ax_recall.set_ylabel(f"Achieved recall@{TOP_K}")
ax_recall.set_title("Asking for more recall returns more")
ax_recall.set_ylim(0, 1.05)
ax_recall.legend()
ax_recall.grid(True, linestyle="--", alpha=0.5)

ax_latency.plot(
    df_metrics["target_recall"],
    df_metrics["median_search_ms"],
    marker="o",
    color="#1a73e8",
)
ax_latency.set_xlabel("Requested target_recall")
ax_latency.set_ylabel("Median search time (ms)")
ax_latency.set_title("Higher recall costs more search time")
ax_latency.set_ylim(bottom=0)
ax_latency.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Analyzing the results

**The recall-latency trade-off is non-linear.** Search latency remains relatively flat across moderate target recall values, then increases steeply near the top of the range. Retrieving the final few percent of nearest neighbors requires evaluating significantly more index partitions and candidate vectors. For instance, decreasing `target_recall` from `0.99` to `0.90` roughly cuts search latency in half, whereas further reducing it from `0.90` to `0.50` produces minimal additional speedup. Understanding this curve helps you identify the sweet spot for your workload.

**Recall scales with the request and levels off at a minimum floor.** Below a certain threshold, the index traverses a baseline minimum number of leaves, meaning further lowering the target will not alter results. Additionally, achieved recall often slightly exceeds requested target recall because the indexing engine selects conservative operational parameters to guarantee target delivery.

**Query embedding dominates overall wall-clock latency.** Generating the query embedding takes hundreds of milliseconds, compared to tens of milliseconds for the vector search itself. Measuring end-to-end request duration would mask the latency impact of `target_recall`. When benchmarking vector search on your own datasets, always isolate index search time from embedding inference.


---

# Part 4: Integrating target recall into your project

In production, you interact with Agent Retrieval via the [Vector Search REST API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search). The `targetRecall` parameter is passed inside `denseScannParams` under the `indexHint`:

```json
{
  "vectorSearch": {
    "searchField": "product_embedding",
    "vector": { "values": [0.0245, -0.0408, "..."] },
    "topK": 10,
    "searchHint": {
      "indexHint": {
        "name": "projects/PROJECT/locations/LOCATION/collections/COLLECTION/indexes/INDEX",
        "denseScannParams": {
          "targetRecall": 0.95
        }
      }
    }
  }
}
```

Endpoint (see [Searching for Data Objects](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search)):

```
POST https://vectorsearch.googleapis.com/v1/projects/{project}/locations/{location}/collections/{collection}/dataObjects:search
```

### Python REST API example

The cell below demonstrates how to construct and send this request using `google.auth` and `requests`.

### Key implementation details

- **Configurable per request:** You can dynamically adjust `target_recall` on a per-query basis (e.g., `0.7` for low-latency candidate retrieval and `0.99` for precision-critical ranking) without rebuilding or re-deploying the index.
- **Requires `indexHint`:** `targetRecall` must be nested under `searchHint.indexHint.denseScannParams`. Requests without `indexHint` use system defaults, and requests with `knnHint` execute exact brute-force search.
- **Data Objects:** Vector search queries are executed against [Data Objects](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/data-objects/data-objects) stored within a Collection.
- **REST API support:** Client libraries (such as `google-cloud-vectorsearch` 0.11.3) may not yet include `targetRecall` in `DenseScannParams`. In such cases, use direct REST API calls as shown below.
- **API version differences:** In `v1`, `targetRecall` is the primary tuning parameter under `denseScannParams`. Legacy parameters (`searchLeavesPct` and `initialCandidateCount`) are available only in `v1beta` and cannot be combined with `targetRecall`. Passing `searchLeavesPct` to `v1` results in an `Unknown name "searchLeavesPct"` error.
- **Index tier compatibility:** Supported on both **performance-optimized** and **storage-optimized** index tiers on a best-effort basis.


In [ ]:
import json

# ==============================================================================
# Sample: Calling the Vector Search REST API with targetRecall
#
# This code illustrates the REST request payload and call structure.
# To execute this against your own GCP project:
#
# 1. If running in Colab, authenticate first:
#      from google.colab import auth
#      auth.authenticate_user()
#
# 2. Or if running locally:
#      gcloud auth application-default login
#
# 3. Replace the placeholder IDs below with your actual project & collection.
# ==============================================================================

# 1. Configuration - Replace these placeholders with your actual GCP resource names
PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "us-central1"
COLLECTION_ID = "YOUR_COLLECTION_ID"
INDEX_ID = "YOUR_INDEX_ID"
SEARCH_FIELD = "product_embedding"

# Example query vector (replace with your actual embedding vector)
QUERY_VECTOR = [0.0245, -0.0408, 0.0123]

# 2. Build endpoint URL and search payload containing targetRecall
collection_path = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
)
index_path = f"{collection_path}/indexes/{INDEX_ID}"
endpoint_url = (
    f"https://vectorsearch.googleapis.com/v1/{collection_path}/dataObjects:search"
)

payload = {
    "vectorSearch": {
        "searchField": SEARCH_FIELD,
        "vector": {"values": QUERY_VECTOR},
        "topK": 10,
        "searchHint": {
            "indexHint": {
                "name": index_path,
                "denseScannParams": {
                    "targetRecall": 0.95,
                },
            }
        },
    }
}

print("Constructed Request URL:")
print(f"  {endpoint_url}\n")
print("Constructed Request Payload:")
print(json.dumps(payload, indent=2))

# ==============================================================================
# 3. Execution (Uncomment when running in your authenticated GCP environment)
# ==============================================================================
# import google.auth
# from google.auth.transport.requests import Request
# import requests
#
# credentials, _ = google.auth.default(
#     scopes=["https://www.googleapis.com/auth/cloud-platform"]
# )
# credentials.refresh(Request())
# headers = {
#     "Authorization": f"Bearer {credentials.token}",
#     "Content-Type": "application/json",
# }
#
# response = requests.post(endpoint_url, json=payload, headers=headers)
# response.raise_for_status()
# search_results = response.json()
# print("Search results:", search_results)


---

# Summary

- **Built-in balance:** Agent Retrieval automatically balances recall and latency for every ANN search. Leaving `target_recall` unset is recommended for most applications.
- **Explicit control:** Use `target_recall` when you need fine-grained control: increase it toward `1.0` for maximum precision, or decrease it to lower query latency and boost QPS.
- **Intuitive abstraction:** It replaces low-level parameters (`search_leaves_pct` and `initial_candidate_count`), allowing you to tune performance using recall goals rather than index implementation details.
- **Per-request flexibility:** Applied dynamically at query time via `searchHint.indexHint.denseScannParams.targetRecall`.
- **Requires clustered ANN index:** Applies only to collections with more than ~100,000 embeddings. Smaller collections use brute-force search where recall is always 100%.
- **Index tier support:** Supported on both performance-optimized and storage-optimized index tiers on a best-effort basis.
- **Benchmarking practice:** When evaluating performance on your own data, isolate vector search latency from query embedding generation, randomize query order, warm up the connection, and aggregate results using medians.

## Next steps

- **[Agent Retrieval interactive demo](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/try-it)**: Explore additional datasets and search modalities in the interactive console.
- **[Agent Retrieval overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)**: Learn about Collections, Data Objects, and Indexes.
- **[Collections documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/collections/collections)**: Create and configure Collections.
- **[Data Objects documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/data-objects/data-objects)**: Ingest, update, and manage Data Objects.
- **[Indexes documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/indexes/indexes)**: Manage Collection Indexes for vector search.
- **[Searching for Data Objects](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search)**: Documentation on semantic, text, hybrid, and vector search.
- **[Agent Development Kit (ADK)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/adk)**: Framework for building and deploying AI agents.
- **[Build a travel agent with Agent Retrieval and ADK](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-travel-agent.ipynb)**: Integrate hybrid search into an AI agent as a callable tool.
